# Semantic-local pilot: evaluation and compute evidence

Reusable definitions for the bounded three-arm Food-101 pilot. This notebook imports only explicitly tagged `ahr-export` definitions; running it defines functions and performs no experiment automatically. The main notebook drives the authorized run.

**Inputs:** the frozen run configuration, audited Food-101 train/development membership, immutable initialization states, and validated full-state final training checkpoints. **Outputs:** isolated compute measurements, six frozen-feature linear probes, paired development-image intervals, representation diagnostics, figures, and a concise report. Official test data are never evaluated here.

**Expected runtime:** compute sweep normally a few minutes; six feature/probe evaluations target under 15 minutes. A supplied Unix deadline is checked between compute calls, feature batches, and bounded probe chunks. **Checkpoints:** `checkpoints/<run_id>/probes/<model_id>/`; features under the run artifact directory. **Resume:** rerun the main notebook with exactly the same RUN_ID and frozen configuration; valid feature caches/probe manifests are checked and resumed, never silently replaced. Completed feature/probe identities include the encoder state and sample order. One writer per run is enforced by the driver and one writer per probe by the inherited evaluator.

This is an exploratory single-seed, equal-update comparison. Bootstrap intervals condition on the fitted models and selected development probe epochs; they exclude training-seed and model-selection uncertainty.

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import nbformat
ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).resolve()
_boot_path = ROOT / "fmm_jepa/bootstrap.ipynb"
_boot_ns = {"__file__": str(_boot_path)}
for _cell in nbformat.read(_boot_path, as_version=4).cells:
    if _cell.cell_type == "code" and "ahr-export" in _cell.metadata.get("tags", []):
        exec(compile(_cell.source, str(_boot_path), "exec"), _boot_ns)
_boot_ns["install_notebook_package"](ROOT)

## Shared identity, deadline and model helpers

In [ ]:
from pathlib import Path
import copy, gc, json, math, statistics, time
import numpy as np
import torch
from fmm_jepa.models import JEPA
from fmm_jepa.food101_training import _model_cfg, isolated_rng, validate_training_manifest
from fmm_jepa.food101_evaluation import (
    atomic_torch, encoder_state_hash, extract_features, fit_linear_probe,
    paired_bootstrap, compute_diagnostics, stable_digest, write_csv)
from fmm_jepa.food101_data import EvaluationDataset, preprocessing_descriptor
from fmm_jepa.training import atomic_json, digest_file, parameter_counts, definition_hashes
from fmm_jepa.geometry import clear_geometry_cache

SEMANTIC_KINDS = ("vit", "fmm", "semantic")
SEMANTIC_LABELS = {"vit": "ViT", "fmm": "Original FMM", "semantic": "Semantic-local"}
SEMANTIC_COLORS = {"vit": "#2966A0", "fmm": "#A26822", "semantic": "#5D7146"}

def _deadline_check(deadline, reserve=0.0):
    if deadline is not None and time.time() + reserve >= float(deadline):
        raise TimeoutError("Registered pilot wall-time deadline reached")

def _release_semantic(device):
    clear_geometry_cache()
    gc.collect()
    if torch.device(device).type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.synchronize(device)

def _build_semantic_comparator(config, kind):
    if kind == "semantic":
        from fmm_jepa.semantic_attention import build_semantic_jepa
        cfg = _model_cfg(config, "vit")
        cfg.update(copy.deepcopy(config.get("semantic", {})))
        cfg["kind"] = "semantic"
        return build_semantic_jepa(cfg).cpu()
    if kind not in ("vit", "fmm"):
        raise ValueError("Unknown semantic-pilot architecture: " + kind)
    return JEPA(_model_cfg(config, kind)).cpu()

class _DeadlineDataset:
    def __init__(self, dataset, deadline):
        self.dataset, self.deadline = dataset, deadline
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, index):
        _deadline_check(self.deadline)
        return self.dataset[index]

def _evaluation_identity(config, bundle, initial, training):
    return {"schema": "semantic-local-evaluation/v1", "config": config,
            "dataset_fingerprint": bundle["dataset"].fingerprint,
            "train_ids": list(bundle["train_ids"]), "dev_ids": list(bundle["dev_ids"]),
            "initial_teachers": {kind: initial["states"][kind]["teacher"] for kind in SEMANTIC_KINDS},
            "training_checkpoints": training, "heldout_status": "NOT_RUN"}

def _probe_with_deadline(train_cache, dev_cache, folder, config, model_id, encoder, deadline):
    # A chunk is at most 250 small classifier updates; each call preserves exact
    # sampler/optimizer/RNG continuation through the inherited probe checkpoint.
    completed = None
    chunk_updates = int(config.get("evaluation", {}).get("probe_deadline_chunk_updates", 50))
    if not 1 <= chunk_updates <= 250:
        raise ValueError("Probe deadline chunk must be 1..250 updates")
    while completed is None:
        _deadline_check(deadline, reserve=2.0)
        manifest_path = Path(folder) / model_id / "probe_manifest.json"
        updates = 0
        if manifest_path.exists():
            manifest = json.loads(manifest_path.read_text())
            updates = int(manifest["latest"]["successful_updates"])
        result = fit_linear_probe(train_cache, dev_cache, folder, config, model_id,
                                  encoder=encoder, stop_after_updates=updates + chunk_updates)
        # Correct the inherited Notebook02 handoff metadata without altering
        # checkpoint payloads or the validated identity of this new run.
        probe_manifest = json.loads(manifest_path.read_text())
        probe_manifest["resume_notebook"] = "notebooks/03_Semantic_Local_Attention_Pilot.ipynb"
        probe_manifest["resume_cell"] = "Notebook03 launch-detached-semantic-pilot; identical RUN_ID and frozen config"
        atomic_json(manifest_path, probe_manifest)
        if result.get("status") != "INTERRUPTED":
            completed = result
    return completed

## Isolated compute comparisons

In [ ]:
def benchmark_semantic_pilot(project_root, config, bundle, artifact_dir, deadline=None):
    """Same actual scaffold, FP32 and microbatch; no simultaneous GPU models.

    Timings exclude data/transfer, checkpoints and profiler. Full updates include
    loss/backward/clipping/AdamW/scheduler/EMA. All timed calls include the same
    finite checks. The benchmark uses one microbatch per optimizer update;
    actual accumulated training timing is reported separately by the driver.
    """
    from fmm_jepa.food101_benchmark import _food_benchmark_case, _food_environment
    root, artifact = Path(project_root), Path(artifact_dir)
    directory = artifact / "compute_benchmark"
    directory.mkdir(parents=True, exist_ok=True)
    policy = {"token_counts": [256, 1024], "microbatch": 2, "warmup": 3,
              "repeats": 10, "seed": int(config["seeds"]["model_init"]),
              "case_timeout_seconds": 120, "max_seconds": 300}
    policy.update(copy.deepcopy(config.get("benchmark", {})))
    policy["warmup"] = max(3, int(policy["warmup"]))
    policy["repeats"] = max(10, int(policy["repeats"]))
    if policy["microbatch"] < 1 or not policy["token_counts"]:
        raise ValueError("Positive benchmark microbatch and token counts required")
    if config.get("dtype", "float32") != "float32":
        raise ValueError("This benchmark is registered for FP32")
    identity = {"config": config, "policy": policy,
                "source_hashes": definition_hashes(root), "environment": _food_environment(config)}
    identity_hash = stable_digest(identity)
    manifest_path = directory / "semantic_manifest.json"
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text())
        if manifest.get("identity_hash") != identity_hash:
            raise ValueError("Compute evidence identity changed; use a new run ID")
    else:
        manifest = {"identity_hash": identity_hash, "records": {}}
        atomic_json(manifest_path, manifest)
    started = time.time()
    local_deadline = min(float(deadline) if deadline is not None else math.inf,
                         started + float(policy["max_seconds"]))
    rows, counts = [], {}
    with isolated_rng(int(policy["seed"])):
        for tokens in policy["token_counts"]:
            for scope in ("encoder_forward", "full_jepa_update"):
                for kind in SEMANTIC_KINDS:
                    key = f"{int(tokens)}_{scope}_{kind}"
                    path = directory / ("semantic_case_" + key + ".json")
                    if key in manifest["records"]:
                        if digest_file(path) != manifest["records"][key]["sha256"]:
                            raise ValueError("Compute case hash changed")
                        row = json.loads(path.read_text())
                    else:
                        row = {"case_key": key, "model_id": kind, "scope": scope,
                               "tokens": int(tokens), "status": "NOT_RUN_DEADLINE"}
                        model = None
                        try:
                            _deadline_check(local_deadline, reserve=5.0)
                            _release_semantic(config["device"])
                            # The only live model begins on CPU; _food_benchmark_case
                            # makes one GPU copy and releases it on completion.
                            with isolated_rng(int(policy["seed"])):
                                model = _build_semantic_comparator(config, kind)
                            counts[kind] = parameter_counts(model)
                            case_policy = dict(policy)
                            case_policy["case_timeout_seconds"] = min(
                                float(policy["case_timeout_seconds"]), local_deadline - time.time())
                            row = _food_benchmark_case(kind, scope, int(tokens), config,
                                                       {kind: model}, case_policy)
                            row["parameter_counts"] = counts[kind]
                            row["case_key"] = key
                        except (RuntimeError, FloatingPointError, TimeoutError, AssertionError) as exc:
                            status = ("OOM" if isinstance(exc, torch.cuda.OutOfMemoryError)
                                      or "out of memory" in str(exc).lower()
                                      else "NOT_RUN_DEADLINE" if isinstance(exc, TimeoutError)
                                      else "ERROR")
                            row.update(status=status, error=repr(exc))
                        finally:
                            model = None
                            _release_semantic(config["device"])
                        row["identity_hash"] = identity_hash
                        atomic_json(path, row)
                        manifest["records"][key] = {"sha256": digest_file(path), "path": str(path)}
                        atomic_json(manifest_path, manifest)
                    counts[kind] = row.get("parameter_counts", counts.get(kind, {}))
                    rows.append(row)
                    print(f"Compute {key}: {row['status']}", flush=True)
    for tokens in policy["token_counts"]:
        for scope in ("encoder_forward", "full_jepa_update"):
            group = [r for r in rows if r["tokens"] == tokens and r["scope"] == scope and r["status"] == "PASS"]
            if len(group) > 1:
                for field in ("image_tensor_sha256", "mask_ids", "augmentation_ids"):
                    if any(row[field] != group[0][field] for row in group[1:]):
                        raise AssertionError("Compute fixture mismatch: " + field)
    result = {"schema": "semantic-local-compute/v1", "identity": identity,
              "identity_hash": identity_hash, "policy": policy, "rows": rows,
              "parameter_counts": counts, "wall_seconds": time.time() - started,
              "status": "PASS" if all(r["status"] == "PASS" for r in rows) else "INCOMPLETE",
              "quality_evidence": False,
              "memory_caveat": "One GPU model per case. Total peak includes its live model/optimizer/input/cache; incremental peak subtracts post-warmup live allocation. CUDA context/reserved allocator are excluded. Preexisting allocation is recorded; inspect it before comparing.",
              "scope": "Synthetic RGB compute only; actual paired training uses the registered accumulated effective batch. No inference about unmeasured asymptotics or image quality."}
    atomic_json(artifact / "semantic_compute_benchmark.json", result)
    return result

## Frozen representations, probes and paired uncertainty

In [ ]:
def evaluate_semantic_pilot(project_root, config, bundle, run_id, artifact_dir, deadline=None):
    root, artifact = Path(project_root), Path(artifact_dir)
    artifact.mkdir(parents=True, exist_ok=True)
    if config.get("evaluation", {}).get("heldout_enabled", False):
        raise ValueError("Official test must remain disabled for this exploratory pilot")
    if set(bundle["train_ids"]) & set(bundle["dev_ids"]):
        raise ValueError("Training/development membership overlap")
    initial_path = artifact / "initialization" / "manifest.json"
    initial = json.loads(initial_path.read_text())
    checkpoint_records = {}
    for kind in SEMANTIC_KINDS:
        folder = root / "checkpoints" / run_id / kind
        manifest = validate_training_manifest(folder)
        checkpoint_records[kind] = {"path": str(folder / manifest["latest"]),
                                    "sha256": digest_file(folder / manifest["latest"]),
                                    "successful_updates": int(manifest["successful_updates"])}
        record = initial["states"][kind]["teacher"]
        if digest_file(record["path"]) != record["sha256"]:
            raise ValueError("Immutable initial teacher hash mismatch")
    budgets = {entry["successful_updates"] for entry in checkpoint_records.values()}
    if len(budgets) != 1:
        raise ValueError("Unequal trained update budgets; comparison refused")
    identity = _evaluation_identity(config, bundle, initial, checkpoint_records)
    identity_hash = stable_digest(identity)
    path = artifact / "semantic_evaluation_manifest.json"
    if path.exists():
        manifest = json.loads(path.read_text())
        if manifest.get("identity_hash") != identity_hash:
            raise ValueError("Evaluation identity changed; never silently reuse")
    else:
        manifest = {"identity_hash": identity_hash, "identity": identity, "completed": {}}
        atomic_json(path, manifest)
    probes, diagnostics, performance, incomplete = {}, {}, [], []
    started = time.time()
    for kind in SEMANTIC_KINDS:
        for control in ("untrained", "trained"):
            model_id = kind + "_" + control
            summary_path = artifact / (model_id + "_semantic_probe.json")
            if model_id in manifest["completed"]:
                entry = manifest["completed"][model_id]
                if digest_file(summary_path) != entry["sha256"]:
                    raise ValueError("Completed evaluation evidence hash mismatch")
                saved = json.loads(summary_path.read_text())
                if saved["identity_hash"] != identity_hash:
                    raise ValueError("Saved evaluation identity mismatch")
                probes[model_id] = saved["probe"]
                diagnostics[model_id] = saved["diagnostics"]
                performance.extend(saved["performance"])
                continue
            model = encoder = None
            try:
                _deadline_check(deadline, reserve=5.0)
                _release_semantic(config["device"])
                model = _build_semantic_comparator(config, kind)
                if control == "untrained":
                    state = torch.load(initial["states"][kind]["teacher"]["path"], map_location="cpu", weights_only=True)
                    model.target.load_state_dict(state)
                else:
                    payload = torch.load(checkpoint_records[kind]["path"], map_location="cpu", weights_only=False)
                    model.load_state_dict(payload["model"])
                    del payload
                encoder = model.target
                del model
                model = None
                # Keep canonical public kind consistent with cache provenance.
                if getattr(encoder, "kind", kind) != kind:
                    raise ValueError("Factory encoder kind disagrees with run architecture")
                common = {"dataset_revision": bundle["dataset"].fingerprint,
                          "checkpoint_hash": encoder_state_hash(encoder), "encoder_kind": kind,
                          "preprocessing": preprocessing_descriptor(config),
                          "pooling": "mean_final_layernorm_patches",
                          "normalization": "encoder final affine LayerNorm then patch mean; train-only per-feature probe standardization"}
                caches, model_performance = {}, []
                for split, ids in (("train", bundle["train_ids"]), ("development", bundle["dev_ids"])):
                    _deadline_check(deadline, reserve=2.0)
                    dataset = _DeadlineDataset(EvaluationDataset(bundle["dataset"], ids, config), deadline)
                    print(f"{model_id}: {len(ids)} frozen {split} features", flush=True)
                    cache = extract_features(encoder, dataset, artifact / "semantic_features" / model_id,
                                             {**common, "sample_ids": list(ids)},
                                             batch_size=int(config["evaluation"]["batch_size"]), device=config["device"])
                    caches[split] = cache
                    perf = cache["performance"]
                    if cache["reused"]:
                        perf = {**json.loads(Path(cache["path"]).with_suffix(".json").read_text())["performance"],
                                "historical_measurement": True, "current_invocation": "valid_cache_reused"}
                    model_performance.append({**perf, "model_id": model_id, "split": split})
                _deadline_check(deadline, reserve=2.0)
                diagnostic_ids = list(bundle["diagnostic_ids"])
                diag_data = _DeadlineDataset(EvaluationDataset(bundle["dataset"], diagnostic_ids, config), deadline)
                encoder.to(config["device"])
                diagnostics[model_id] = compute_diagnostics(encoder, diag_data, config)
                encoder.cpu()
                _release_semantic(config["device"])
                result = _probe_with_deadline(caches["train"], caches["development"],
                    root / "checkpoints" / run_id / "probes", config, model_id, encoder, deadline)
                probes[model_id] = {"summary": result["summary"], "metrics": result["metrics"],
                                    "predictions": result["predictions"]}
                model_performance.append({**result["performance"], "model_id": model_id,
                                          "scope_note": "Final bounded probe chunk only; whole evaluation wall includes all chunks."})
                performance.extend(model_performance)
                write_csv(artifact / "semantic_predictions" / (model_id + "_development.csv"), result["predictions"])
                saved = {"identity_hash": identity_hash, "probe": probes[model_id],
                         "diagnostics": diagnostics[model_id], "performance": model_performance}
                atomic_json(summary_path, saved)
                manifest["completed"][model_id] = {"path": str(summary_path), "sha256": digest_file(summary_path)}
                atomic_json(path, manifest)
                print(f"{model_id}: dev top-1 {100 * result['summary']['metrics']['top1']:.2f}%", flush=True)
            except TimeoutError as exc:
                incomplete.append({"model_id": model_id, "status": "INCOMPLETE_DEADLINE", "reason": str(exc)})
                print(f"{model_id}: evaluation stopped at registered deadline", flush=True)
            finally:
                encoder = model = None
                _release_semantic(config["device"])
    if probes:
        initialization_hashes = {p["summary"]["initialization_hash"] for p in probes.values()}
        probe_budgets = [p["summary"]["budget"] for p in probes.values()]
        if len(initialization_hashes) != 1 or any(p != probe_budgets[0] for p in probe_budgets):
            raise AssertionError("Probe initializations or budgets are unmatched")
    contrasts = {"learning_" + kind: (kind + "_trained", kind + "_untrained") for kind in SEMANTIC_KINDS}
    contrasts.update(semantic_vs_vit=("semantic_trained", "vit_trained"),
                     semantic_vs_fmm=("semantic_trained", "fmm_trained"),
                     fmm_vs_vit=("fmm_trained", "vit_trained"))
    comparisons = {}
    for name, (first, second) in contrasts.items():
        if first in probes and second in probes:
            comparisons[name] = {"first": first, "second": second,
                "top1": paired_bootstrap(probes[first]["predictions"], probes[second]["predictions"],
                    repeats=int(config["probe"].get("bootstrap_repeats", 2000)),
                    seed=int(config["seeds"]["bootstrap"]), metric="correct_top1")}
    result = {"schema": "semantic-local-evaluation/v1", "run_id": run_id,
              "status": "COMPLETED" if len(probes) == 6 else "INCOMPLETE",
              "identity_hash": identity_hash, "completed_models": list(probes),
              "probes": {k: v["summary"] for k, v in probes.items()},
              "diagnostics": diagnostics, "contrasts": comparisons,
              "performance": performance, "incomplete": incomplete,
              "actual_successful_updates": next(iter(budgets)),
              "planned_successful_updates": int(config["training"]["updates"]),
              "evaluation_wall_seconds": time.time() - started,
              "training_seeds": 1, "heldout_status": "NOT_RUN",
              "full_grid_tokens": (int(config["image_size"]) // int(config["model"]["patch_size"])) ** 2,
              "uncertainty": "Paired development-image percentile bootstrap. Conditional on these trained weights and development-selected probe epochs. Excludes seed and selection uncertainty; multiple exploratory contrasts are not multiplicity-adjusted.",
              "comparison": "Same scaffold, visible images, update count, augmentations and masks; parameters reported explicitly; no robust architecture superiority claim from one seed."}
    result["screening"] = semantic_screening(artifact, result)
    atomic_json(artifact / "semantic_evaluation.json", result)
    atomic_json(artifact / "semantic_heldout_results.json", {"status": "NOT_RUN", "reason": "Official test remains reserved"})
    write_csv(artifact / "semantic_probe_metrics.csv", [row for p in probes.values() for row in p["metrics"]])
    write_semantic_report(artifact, config, result)
    return result

## Readable report and paired-comparison figures

In [ ]:
def semantic_screening(artifact_dir, evaluation):
    """Thresholds fixed before outcomes; independent cost/learning/quality decisions."""
    artifact = Path(artifact_dir)
    benchmark_path = artifact / "semantic_compute_benchmark.json"
    compute = json.loads(benchmark_path.read_text()) if benchmark_path.exists() else {"rows": []}
    ratios = []
    for semantic in compute["rows"]:
        if semantic.get("status") != "PASS" or semantic["model_id"] != "semantic":
            continue
        for other in ("vit", "fmm"):
            comparison = next((r for r in compute["rows"] if r.get("status") == "PASS"
                and r["model_id"] == other and r["scope"] == semantic["scope"] and r["tokens"] == semantic["tokens"]), None)
            if comparison is None:
                continue
            row = {"candidate": "semantic", "comparator": other,
                   "scope": semantic["scope"], "tokens": semantic["tokens"],
                   "median_seconds_ratio": semantic["median_seconds"] / comparison["median_seconds"]}
            for field in ("peak_allocated_bytes", "incremental_peak_bytes"):
                a, b = semantic.get(field), comparison.get(field)
                row[field + "_ratio"] = a / b if a is not None and b is not None and b > 0 else None
            ratios.append(row)
    train_path = artifact / "training_summary.json"
    training = json.loads(train_path.read_text()) if train_path.exists() else {}
    train_ratios = {}
    records = training.get("results", {})
    candidate = records.get("semantic", {}).get("performance", {})
    for other in ("vit", "fmm"):
        baseline = records.get(other, {}).get("performance", {})
        a = (candidate.get("complete_jepa_step_seconds") or {}).get("median")
        b = (baseline.get("complete_jepa_step_seconds") or {}).get("median")
        if a is not None and b is not None and b > 0:
            train_ratios[other] = {"median_complete_training_step_ratio": a / b,
                "scope": "Actual accumulated training, paired config and successful update budget"}
    fixed_tokens = int(round(math.sqrt(int(evaluation.get("full_grid_tokens", 256))))) ** 2
    old_fmm_ratio = train_ratios.get("fmm", {}).get("median_complete_training_step_ratio")
    cost_scope = "actual_accumulated_training"
    if old_fmm_ratio is None:
        match = next((r for r in ratios if r["comparator"] == "fmm" and r["scope"] == "full_jepa_update"
                      and r["tokens"] == fixed_tokens), None)
        old_fmm_ratio = match["median_seconds_ratio"] if match else None
        cost_scope = "synthetic_single_microbatch_update_at_training_grid"
    learning = evaluation["contrasts"].get("learning_semantic", {}).get("top1")
    low_rank = evaluation["diagnostics"].get("semantic_trained", {})
    degeneration = bool(low_rank.get("zero_variance_flag", False) or low_rank.get("very_low_rank_flag", False))
    learning_status = ("NOT_MEASURED" if learning is None else "learning_promising"
        if learning["ci95_percentage_points"][0] > 0 and not degeneration else "learning_unresolved_or_concerning")
    quality = {}
    for contrast in ("semantic_vs_vit", "semantic_vs_fmm"):
        value = evaluation["contrasts"].get(contrast, {}).get("top1")
        if value is None:
            quality[contrast] = "NOT_MEASURED"
        else:
            lo, hi = value["ci95_percentage_points"]
            quality[contrast] = ("positive_for_this_paired_seed" if lo > 0 else
                                 "negative_for_this_paired_seed" if hi < 0 else "quality_unresolved")
    return {"preregistered_cost_threshold": "At least 20% lower median complete step time than original FMM (ratio <= 0.8). Actual accumulated training preferred; synthetic scope explicit if used.",
            "cost_status": "NOT_MEASURED" if old_fmm_ratio is None else "cost_promising" if old_fmm_ratio <= .8 else "cost_threshold_not_met",
            "cost_scope": cost_scope, "semantic_to_old_fmm_step_ratio": old_fmm_ratio,
            "learning_status": learning_status, "semantic_degeneracy_advisory": degeneration,
            "quality_status": quality, "compute_ratios": ratios, "actual_training_ratios": train_ratios,
            "interpretation": "Separate exploratory decisions. No universal success score, no noninferiority inferred from an interval spanning zero, and no training-seed uncertainty estimated."}


def write_semantic_report(artifact_dir, config, evaluation):
    """Render inspectable static figures and Markdown from saved numerical evidence."""
    import matplotlib
    matplotlib.use("Agg", force=True)
    import matplotlib.pyplot as plt
    from matplotlib.ticker import MaxNLocator
    artifact = Path(artifact_dir)
    compute_path = artifact / "semantic_compute_benchmark.json"
    compute = json.loads(compute_path.read_text()) if compute_path.exists() else {"rows": []}
    training_path = artifact / "training_summary.json"
    training = json.loads(training_path.read_text()) if training_path.exists() else {}
    plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
                         "figure.facecolor": "white", "axes.titleweight": "bold"})
    figures = []
    probes = evaluation["probes"]
    if probes:
        fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.5), constrained_layout=True)
        xs = np.arange(3)
        for index, control in enumerate(("untrained", "trained")):
            values = [100 * probes[kind + "_" + control]["metrics"]["top1"]
                      if kind + "_" + control in probes else np.nan for kind in SEMANTIC_KINDS]
            axes[0].bar(xs + (index - .5) * .32, values, width=.30,
                        color=[SEMANTIC_COLORS[k] for k in SEMANTIC_KINDS],
                        alpha=.40 if index == 0 else 1.0,
                        hatch="//" if index == 0 else None, edgecolor="#303030",
                        label="Initial" if index == 0 else "Trained")
            for x, value in zip(xs + (index - .5) * .32, values):
                if np.isfinite(value): axes[0].text(x, value + .15, f"{value:.2f}", ha="center", fontsize=10)
        axes[0].set_xticks(xs, [SEMANTIC_LABELS[k] for k in SEMANTIC_KINDS])
        axes[0].set_ylabel("Development top-1 accuracy (%)")
        axes[0].set_ylim(bottom=0)
        axes[0].set_title("Frozen-feature linear probes")
        axes[0].legend(frameon=False)
        axes[0].grid(axis="y", color="#e6e6e6"); axes[0].set_axisbelow(True)
        names = list(evaluation["contrasts"])
        labels = {"learning_vit": "ViT: trained − initial", "learning_fmm": "FMM: trained − initial",
                  "learning_semantic": "Semantic-local: trained − initial", "semantic_vs_vit": "Semantic-local − ViT",
                  "semantic_vs_fmm": "Semantic-local − FMM", "fmm_vs_vit": "FMM − ViT"}
        for i, name in enumerate(names):
            d = evaluation["contrasts"][name]["top1"]
            lo, hi = d["ci95_percentage_points"]; mid = d["delta_percentage_points"]
            axes[1].plot([lo, hi], [i, i], color="#2966A0", linewidth=2)
            axes[1].plot(mid, i, "o", color="#2966A0")
        axes[1].set_yticks(range(len(names)), [labels[n] for n in names])
        axes[1].invert_yaxis(); axes[1].axvline(0, color="#303030", linewidth=1)
        axes[1].set_xlabel("Top-1 difference (percentage points)")
        axes[1].set_title("Paired 95% image-bootstrap intervals")
        fig.suptitle(f"Food-101 development · {evaluation['actual_successful_updates']} updates · one paired seed", fontsize=13)
        fig.savefig(artifact / "semantic_quality.png", dpi=160); plt.close(fig)
        figures.append("semantic_quality.png")
    rows = [r for r in compute["rows"] if r.get("status") == "PASS"]
    if rows:
        fig, axes = plt.subplots(2, 2, figsize=(11.5, 8), constrained_layout=True)
        for column, scope in enumerate(("encoder_forward", "full_jepa_update")):
            for kind in SEMANTIC_KINDS:
                selected = sorted([r for r in rows if r["scope"] == scope and r["model_id"] == kind], key=lambda r:r["tokens"])
                if not selected: continue
                tokens = [r["tokens"] for r in selected]
                for row_index, field, scale in ((0, "median_seconds", 1000), (1, "peak_allocated_bytes", 1 / 1024**2)):
                    values = [r.get(field) for r in selected]
                    axes[row_index, column].plot(tokens, [v * scale if v is not None else np.nan for v in values],
                        marker={"vit":"o", "fmm":"s", "semantic":"^"}[kind], color=SEMANTIC_COLORS[kind], label=SEMANTIC_LABELS[kind])
                    axes[row_index, column].set_xticks(sorted({r["tokens"] for r in rows}))
                    axes[row_index, column].set_ylim(bottom=0)
                    axes[row_index, column].grid(color="#e6e6e6")
            axes[0, column].set_title("Encoder forward" if scope == "encoder_forward" else "Full JEPA update (one microbatch)")
            axes[0, column].set_ylabel("Median synchronized time (ms)")
            axes[1, column].set_ylabel("Total peak CUDA allocation (MiB)")
            axes[1, column].set_xlabel("Full-grid patch tokens")
            axes[0, column].legend(frameon=False)
        fig.suptitle("Synthetic compute · identical FP32 scaffold and batch · isolated model allocations", fontsize=12)
        fig.savefig(artifact / "semantic_compute.png", dpi=160); plt.close(fig)
        figures.append("semantic_compute.png")
    lines = ["# Semantic-local exploratory Food-101 pilot", "",
             f"Run: `{evaluation['run_id']}`. Status: **{evaluation['status']}**. One paired training seed.", "",
             f"Each architecture reached {evaluation['actual_successful_updates']} successful updates; the registered plan was {evaluation['planned_successful_updates']}. Official test: **NOT_RUN**.", "",
             "The candidate replaces spatial FMM mixing with normalized positive semantic features and exact nearby exponential weights. This is a kernel-attention candidate, not a validated softmax approximation bound or a spatial FMM speedup.", "",
             "| Representation | Development top-1 | Development top-5 |", "|---|---:|---:|"]
    for model_id, p in probes.items():
        kind, control = model_id.rsplit("_", 1)
        label = SEMANTIC_LABELS[kind] + (" initial" if control == "untrained" else " trained")
        lines.append(f"| {label} | {100*p['metrics']['top1']:.2f}% | {100*p['metrics']['top5']:.2f}% |")
    lines += ["", "| Paired top-1 contrast | Difference (pp) | 95% image-bootstrap interval (pp) |", "|---|---:|---:|"]
    for name, value in evaluation["contrasts"].items():
        d = value["top1"]; lo, hi = d["ci95_percentage_points"]
        lines.append(f"| {name} | {d['delta_percentage_points']:+.2f} | [{lo:+.2f}, {hi:+.2f}] |")
    lines += ["", "| Representation | Raw pooled effective rank | Mean feature std | Degeneracy advisory |", "|---|---:|---:|---|"]
    for model_id, d in evaluation["diagnostics"].items():
        lines.append(f"| {model_id} | {d['effective_rank']:.2f}/{d['effective_rank_ceiling']} | {d['feature_std_mean']:.6f} | {d['zero_variance_flag'] or d['very_low_rank_flag']} |")
    lines += ["", "| Grid tokens | Scope | Model | Median ms | Total peak MiB | Incremental peak MiB |", "|---:|---|---|---:|---:|---:|"]
    for row in rows:
        total = row.get("peak_allocated_bytes"); incremental = row.get("incremental_peak_bytes")
        total_text = f"{total/1024**2:.2f}" if total is not None else "UNKNOWN"
        inc_text = f"{incremental/1024**2:.2f}" if incremental is not None else "UNKNOWN"
        lines.append(f"| {row['tokens']} | {row['scope']} | {SEMANTIC_LABELS[row['model_id']]} | {1000*row['median_seconds']:.3f} | {total_text} | {inc_text} |")
    for name in figures:
        lines += ["", f"![{name.replace('_', ' ').replace('.png', '')}]({name})"]
    screen = evaluation.get("screening", {})
    lines += ["", f"Predeclared screening: cost **{screen.get('cost_status', 'NOT_MEASURED')}**; learning **{screen.get('learning_status', 'NOT_MEASURED')}**.",
              "", screen.get("preregistered_cost_threshold", ""),
              "", "Quality contrast status: " + json.dumps(screen.get("quality_status", {}), sort_keys=True),
              "", "| Candidate / comparator | Scope | Median step ratio |", "|---|---|---:|"]
    for other, row in screen.get("actual_training_ratios", {}).items():
        lines.append(f"| Semantic-local / {SEMANTIC_LABELS[other]} | Actual accumulated training | {row['median_complete_training_step_ratio']:.3f} |")
    lines += ["", "Interpretation rules: trained-minus-initial tests whether this short schedule improved the frozen representation. Candidate-minus-comparator intervals describe this paired seed only. An interval spanning zero is unresolved, not equivalence. Favorable compute without learning is insufficient; favorable development accuracy without favorable cost does not establish efficiency.", "",
              evaluation["uncertainty"], "",
              "The model scaffold, training exposures, masks and augmentation streams are shared. Parameter counts and timing scopes are explicit in semantic_compute_benchmark.json. The linear probe standardizes using training features only, shares its initialization/order/budget, and selects the best development epoch. Geometry and representation caches are cleared between GPU models. Compute timing includes finite checks; synthetic noise provides no learning evidence. Total and incremental CUDA memory exclude reserved allocator/context memory.", "",
              "No error bound for this finite random feature realization has been established. The operator certificates test algebra, mask behavior, derivatives and checkpoint recovery on bounded cases. Training is a small exploratory screen, not a definitive architecture ranking.", "",
              "Resume: rerun the main semantic pilot notebook with this RUN_ID and identical frozen configuration. Training/probe checkpoints retain exact recovery state and the latest two valid snapshots plus best. Feature caches use full state/membership identities. A deadline cutoff remains explicitly incomplete; extending a deadline requires user authorization."]
    if evaluation["incomplete"]:
        lines += ["", "Incomplete evaluations:", ""] + ["- " + x["model_id"] + ": " + x["reason"] for x in evaluation["incomplete"]]
    (artifact / "semantic_results.md").write_text("\n".join(lines) + "\n")
    atomic_json(artifact / "semantic_figure_manifest.json", {"figures": figures,
                "source_evaluation": "semantic_evaluation.json", "source_compute": "semantic_compute_benchmark.json",
                "visual_inspection": "PENDING_human_or_agent_image_review"})
    return {"report": str(artifact / "semantic_results.md"), "figures": [str(artifact / p) for p in figures]}

## Import-only validation

All reusable functions above are tagged `ahr-export`. This support notebook defines evidence generation; it does not start training, feature extraction or benchmarks. The authorized main driver supplies the frozen configuration and absolute deadline. Figures are saved beside their source JSON and must be visually inspected before final handoff.

In [ ]:
assert set(SEMANTIC_LABELS) == set(SEMANTIC_KINDS)
assert callable(benchmark_semantic_pilot) and callable(evaluate_semantic_pilot)
print("Semantic-local evaluation definitions loaded; no experiment started.")